# Bidirectional RNNs
1. Parameter counts: SimpleRNN, LSTM and GRU, alone and inside Keras' Bidirectional wrapper.
2. A check of the equations: Bidirectional = a forward RNN and a backward RNN, outputs joined at every step.
3. Experiment: part-of-speech tagging of real Wall Street Journal sentences (CoNLL-2000). A tag at one word often
   depends on the next words. Unidirectional LSTM vs bidirectional LSTM vs a unidirectional LSTM with the same
   number of parameters as the bidirectional one. 3 seeds each.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import gzip
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()   # same numbers on every run
print("GPU:", tf.config.list_physical_devices("GPU"))
DATA = Path("data")

## 1. Parameters double inside the Bidirectional wrapper

In [ ]:
rows = []
for name, layer in (("SimpleRNN", keras.layers.SimpleRNN), ("LSTM", keras.layers.LSTM), ("GRU", keras.layers.GRU)):
    one = keras.Sequential([keras.Input(shape=(100,)), keras.layers.Embedding(10000, 32), layer(5),
                            keras.layers.Dense(1, activation="sigmoid")])
    two = keras.Sequential([keras.Input(shape=(100,)), keras.layers.Embedding(10000, 32),
                            keras.layers.Bidirectional(layer(5)), keras.layers.Dense(1, activation="sigmoid")])
    rows.append(dict(layer=name, unidirectional=one.layers[1].count_params(),
                     bidirectional=two.layers[1].count_params(), dense_uni=one.layers[2].count_params(),
                     dense_bi=two.layers[2].count_params()))
params = pd.DataFrame(rows)
print(params.to_string(index=False))
assert (params.bidirectional == 2 * params.unidirectional).all() and params.unidirectional[0] == 190
params.to_csv(DATA / "params.csv", index=False)

## 2. What the wrapper computes
Forward RNN reads x_1..x_T, backward RNN reads x_T..x_1; at step t the output is [forward h_t, backward h_t].

In [ ]:
x = np.random.default_rng(0).normal(size=(1, 4, 3)).astype("float32")      # 1 sequence, 4 steps, 3 features
bi = keras.layers.Bidirectional(keras.layers.SimpleRNN(2, return_sequences=True))
out = bi(x).numpy()[0]
Wi_f, Wh_f, b_f, Wi_b, Wh_b, b_b = bi.get_weights()


def run(seq, Wi, Wh, b):
    h, hs = np.zeros(2), []
    for x_t in seq:
        h = np.tanh(x_t @ Wi + h @ Wh + b)
        hs.append(h)
    return np.array(hs)


fwd = run(x[0], Wi_f, Wh_f, b_f)                      # left to right
bwd = run(x[0][::-1], Wi_b, Wh_b, b_b)[::-1]          # right to left, then put back in time order
print("Keras output shape:", out.shape, "(time steps, 2 forward + 2 backward)")
assert np.allclose(out, np.concatenate([fwd, bwd], axis=1), atol=1e-5)
print("matches [forward h_t, backward h_t] at every step")
print("backward h_1 depends on every word: it is the backward RNN's LAST state")

## 3. Part-of-speech tagging on CoNLL-2000
8,936 training and 2,012 test sentences from the Wall Street Journal, each word labelled with one of 44 tags.

In [ ]:
def read(path):
    sents, cur = [], []
    for line in gzip.open(path, "rt"):
        p = line.split()
        if p:
            cur.append((p[0], p[1]))
        elif cur:
            sents.append(cur)
            cur = []
    return sents + ([cur] if cur else [])


train, test = read(DATA / "train.wordpos.gz"), read(DATA / "test.wordpos.gz")
print(len(train), "training sentences,", len(test), "test sentences")
print(" ".join(f"{w}/{t}" for w, t in train[0]))

word_counts = Counter(w for s in train for w, _ in s)
words = [w for w, c in word_counts.most_common() if c >= 2]          # rarer words become the unknown word
word_id = {w: i + 2 for i, w in enumerate(words)}                    # 0 = padding, 1 = unknown
tags = sorted({t for s in train for _, t in s})
tag_id = {t: i for i, t in enumerate(tags)}
MAXLEN = 80                                                          # longest sentence has 78 words


def encode(sents):
    X = np.zeros((len(sents), MAXLEN), "int32")
    Y = np.zeros((len(sents), MAXLEN), "int32")
    W = np.zeros((len(sents), MAXLEN), "float32")                    # 1 for real words, 0 for padding
    for i, s in enumerate(sents):
        for j, (w, t) in enumerate(s):
            X[i, j], Y[i, j], W[i, j] = word_id.get(w, 1), tag_id.get(t, 0), 1
    return X, Y, W


Xtr, Ytr, Wtr = encode(train)
Xte, Yte, Wte = encode(test)
print("vocabulary:", len(word_id) + 2, "| tags:", len(tags), "| X shape:", Xtr.shape)

In [ ]:
def tagger(kind):
    rnn = {"unidirectional LSTM, 64 nodes": keras.layers.LSTM(64, return_sequences=True),
           "bidirectional LSTM, 64 + 64 nodes": keras.layers.Bidirectional(keras.layers.LSTM(64, return_sequences=True)),
           "unidirectional LSTM, 100 nodes (same parameters)": keras.layers.LSTM(100, return_sequences=True)}[kind]
    return keras.Sequential([keras.Input(shape=(MAXLEN,)), keras.layers.Embedding(len(word_id) + 2, 64),
                             rnn, keras.layers.Dense(len(tags), activation="softmax")])   # one tag per word


KINDS = ["unidirectional LSTM, 64 nodes", "bidirectional LSTM, 64 + 64 nodes",
         "unidirectional LSTM, 100 nodes (same parameters)"]
for k in KINDS:
    print(f"{k:50} recurrent parameters {tagger(k).layers[1].count_params():,}")

In [ ]:
EPOCHS, SEEDS = 8, [1, 2, 3]
hist, preds = [], {}
for k in KINDS:
    for s in SEEDS:
        keras.utils.set_random_seed(s)
        m = tagger(k)
        m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", weighted_metrics=["accuracy"])
        h = m.fit(Xtr, Ytr, sample_weight=Wtr, validation_data=(Xte, Yte, Wte), epochs=EPOCHS, batch_size=64,
                  verbose=0)
        for e in range(EPOCHS):
            hist.append(dict(model=k, seed=s, epoch=e + 1, accuracy=h.history["accuracy"][e],
                             val_accuracy=h.history["val_accuracy"][e]))
        preds[k, s] = m.predict(Xte, verbose=0).argmax(-1)
        print(f"{k:50} seed {s}: test accuracy {h.history['val_accuracy'][-1]:.4f}")
hist = pd.DataFrame(hist)
hist.round(5).to_csv(DATA / "pos_history.csv", index=False)

In [ ]:
# Test accuracy over real words only, computed directly from the predictions
mask = Wte == 1
final = pd.DataFrame([dict(model=k, seed=s, test_accuracy=(preds[k, s] == Yte)[mask].mean()) for (k, s) in preds])
summary = final.groupby("model", sort=False).test_accuracy.agg(["mean", "min", "max"])
print(summary.round(4))

## Where the gain comes from: words whose tag depends on what follows
A word is "ambiguous" if, in the training data, its second most common tag covers at least 10% of its uses.

In [ ]:
tag_counts = defaultdict(Counter)
for s in train:
    for w, t in s:
        tag_counts[w.lower()][t] += 1
ambiguous = {w for w, c in tag_counts.items() if len(c) > 1 and c.most_common(2)[1][1] >= 0.1 * sum(c.values())}
top6 = sorted(ambiguous, key=lambda w: -sum(tag_counts[w].values()))[:6]           # the 6 most frequent ones
print("most frequent ambiguous words:", top6)
lower = np.array([[w.lower() for w, _ in s] + [""] * (MAXLEN - len(s)) for s in test])
is_amb = np.isin(lower, list(ambiguous)) & mask

rows = []
for k in KINDS:
    acc = lambda sel: np.mean([(preds[k, s] == Yte)[sel].mean() for s in SEEDS])
    row = dict(model=k, all_words=acc(mask), ambiguous_words=acc(is_amb), other_words=acc(mask & ~is_amb))
    row.update({w: acc(lower == w) for w in top6})
    rows.append(row)
by_word = pd.DataFrame(rows)
print("share of test words that are ambiguous:", round(is_amb.sum() / mask.sum(), 3))
print("test count of each:", {w: int((lower == w).sum()) for w in top6})
print(by_word.round(3).T)
by_word.round(4).to_csv(DATA / "pos_by_word.csv", index=False)

In [ ]:
# One sentence, tagged by the unidirectional and the bidirectional model (seed 1)
i = next(i for i, s in enumerate(test) if any(w == "'s" for w, _ in s) and len(s) <= 14)
for j, (w, t) in enumerate(test[i]):
    u = tags[preds[KINDS[0], 1][i, j]]
    b = tags[preds[KINDS[1], 1][i, j]]
    print(f"{w:12} true {t:5} uni {u:5} bi {b:5}{'  <-' if u != b else ''}")